<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/01_study_area_grid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Study area grid

**Purpose:** Build the spatial grid that every later notebook samples from: ~2 km cells covering dryland cropland in nine Sahel countries.

**Study area:** Senegal, Gambia, Mauritania, Mali, Burkina Faso, Niger, Nigeria, Chad, Cameroon (FAO GAUL boundaries).

**Method:** A cell is kept if at least 5% of it is cropland (ESA WorldCover v200, class 40) and its aridity index (precipitation / PET, TerraClimate 1991-2020) is below 0.65. Exported from Google Earth Engine as 1° tiles, merged in Colab, and each point is labelled with its country and state.

**Inputs:** Google Earth Engine (`FAO/GAUL/2015/level1`, `ESA/WorldCover/v200`, `IDAHO_EPSCOR/TERRACLIMATE`). Requires an Earth Engine account and a Cloud project.

**Outputs:**
- `grid.parquet`: one row per cell (`grid_id`, `lon`, `lat`, `crop_frac`, `aridity_index`, `adm0`, `adm1`, `adm1_code`). Stored in Google Drive, not in this repo.
- `states.geojson` / `states.gpkg`: simplified state boundaries (the GeoJSON is in `data/`).

**Notes:** Latitude coverage is 4°N to 25°N. Set `PROJECT_ID` to your own Cloud project before running. Runtime is dominated by waiting for Earth Engine export tasks.

In [ ]:
# @title This cell connects to Earth Engine, checks that all nine countries are found in the boundary dataset, and counts the states or provinces in each.
import ee

# ---------- Settings ----------
PROJECT_ID = 'your-project-id'
COUNTRIES = ['Senegal', 'Gambia', 'Mauritania', 'Mali', 'Burkina Faso',
             'Niger', 'Nigeria', 'Chad', 'Cameroon']

BASELINE = ('1991-01-01', '2021-01-01')   # climate normals (end date exclusive)
RECENT   = ('2018-01-01', '2025-01-01')   # Sentinel-era window (used in later steps)
# ------------------------------

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print('Connected to Earth Engine')

gaul = ee.FeatureCollection('FAO/GAUL/2015/level1')
admin1 = gaul.filter(ee.Filter.inList('ADM0_NAME', COUNTRIES))

found = admin1.aggregate_array('ADM0_NAME').distinct().getInfo()
missing = [c for c in COUNTRIES if c not in found]

print('\nTotal states/provinces:', admin1.size().getInfo())
print('Countries found  :', sorted(found))
print('Countries MISSING:', missing if missing else 'none')

# Count of states per country
counts = admin1.aggregate_histogram('ADM0_NAME').getInfo()
for c in sorted(counts):
    print(f'  {c:15} {int(counts[c])}')

# If anything is missing, show how GAUL spells similar names
if missing:
    all_names = gaul.aggregate_array('ADM0_NAME').distinct().getInfo()
    for m in missing:
        hints = [n for n in all_names if m.split()[0].lower()[:4] in n.lower()]
        print(f'Possible spellings for {m}:', hints)

In [ ]:
# @title This cell build the grid image and test on Gambia; A test so mistakes in the mask show up
# ---------- Grid settings ----------
GRID_SCALE   = 2000    # metres; drop to 1000 later if Colab and quota cope
CROP_MIN     = 0.05    # keep cells with >= 5% cropland (WorldCover under-detects small Sahel fields)
ARIDITY_MAX  = 0.65    # UN drylands threshold; raise to ~0.8 to include more of the Sudan savanna
DRIVE_FOLDER = 'crop_project_grid'
# -----------------------------------

proj = ee.Projection('EPSG:4326').atScale(GRID_SCALE)

# Cropland fraction per cell (WorldCover class 40 = cropland, map year 2021)
wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
crop_frac = (wc.eq(40).float()
             .reduceResolution(ee.Reducer.mean(), maxPixels=65536)
             .reproject(proj)
             .rename('crop_frac'))

# Aridity index = precipitation / PET over the 30-year baseline (TerraClimate)
tc = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').filterDate(*BASELINE)
ai = (tc.select('pr').sum()
        .divide(tc.select('pet').sum().multiply(0.1))   # PET is stored x10
        .rename('aridity_index'))

mask = crop_frac.gte(CROP_MIN).And(ai.lt(ARIDITY_MAX))
grid_img = (crop_frac.addBands(ai)
            .addBands(ee.Image.pixelLonLat())
            .updateMask(mask))

countries_fc = (ee.FeatureCollection('FAO/GAUL/2015/level0')
                .filter(ee.Filter.inList('ADM0_NAME', COUNTRIES)))

def sample_country(name):
    region = countries_fc.filter(ee.Filter.eq('ADM0_NAME', name)).geometry()
    pts = grid_img.sample(region=region, projection=proj, scale=GRID_SCALE,
                          dropNulls=True, tileScale=4)
    return pts.map(lambda ft: ft.set('adm0', name))

# ---- Test on Gambia ----
#test = sample_country('Gambia')
#print('Gambia grid cells:', test.size().getInfo())
#for f in test.limit(3).getInfo()['features']:
#    print(f['properties'])#

In [ ]:
# @title This cell build the tiles for the 9 countries and test one
import re

STEP = 1                           # tile size in degrees (was 2)
LON_RANGE = range(-18, 16, STEP)
LAT_RANGE = range(4, 18, STEP)

target_names = [c for c in COUNTRIES if c != 'Gambia']
target_fc = countries_fc.filter(ee.Filter.inList('ADM0_NAME', target_names))

tile_list = []
for x in LON_RANGE:
    for y in LAT_RANGE:
        geom = ee.Geometry.Rectangle([x, y, x + STEP, y + STEP], 'EPSG:4326', False)
        tile_list.append(ee.Feature(geom, {'name': f'tile_x{x}_y{y}'}))

tiles = ee.FeatureCollection(tile_list).filterBounds(target_fc.geometry())
names = sorted(tiles.aggregate_array('name').getInfo())
print(len(names), 'tiles intersect the target countries')

def tile_points(name):
    x, y = map(int, re.match(r'tile_x(-?\d+)_y(-?\d+)', name).groups())
    geom = ee.Geometry.Rectangle([x, y, x + STEP, y + STEP], 'EPSG:4326', False)
    return grid_img.sample(region=geom, projection=proj, scale=GRID_SCALE,
                           dropNulls=True, tileScale=4)

# Test one tile near Kano, northern Nigeria (covers 8-9 E, 12-13 N)
test_name = 'tile_x8_y12'
print(test_name, 'is in the tile list:', test_name in names)
print('Cells in test tile:', tile_points(test_name).size().getInfo())

In [ ]:
# @title This cell start all exports
for n in names:
    ee.batch.Export.table.toDrive(
        collection=tile_points(n),
        description=n,
        folder=DRIVE_FOLDER,
        fileFormat='CSV',
        selectors=['longitude', 'latitude', 'crop_frac', 'aridity_index']
    ).start()

print(f'Started {len(names)} tasks')

In [ ]:
# @title This cell exports Eatern Chad & Cameroon
import re

STEP = 1   # must match the tile size used earlier

def tile_points(name):
    x, y = map(int, re.match(r'tile_x(-?\d+)_y(-?\d+)', name).groups())
    geom = ee.Geometry.Rectangle([x, y, x + STEP, y + STEP], 'EPSG:4326', False)
    return grid_img.sample(region=geom, projection=proj, scale=GRID_SCALE,
                           dropNulls=True, tileScale=4)

chad_cam = countries_fc.filter(ee.Filter.inList('ADM0_NAME', ['Chad', 'Cameroon']))

extra = [ee.Feature(ee.Geometry.Rectangle([x, y, x + STEP, y + STEP], 'EPSG:4326', False),
                    {'name': f'tile_x{x}_y{y}'})
         for x in range(16, 25) for y in range(6, 18)]

extra_names = sorted(ee.FeatureCollection(extra)
                     .filterBounds(chad_cam.geometry())
                     .aggregate_array('name').getInfo())

# Skip any tile that was already submitted, so re-running is safe
already = {t.config.get('description') for t in ee.batch.Task.list()[:1000]}
todo = [n for n in extra_names if n not in already]
print(len(extra_names), 'extra tiles;', len(todo), 'to start')

for n in todo:
    ee.batch.Export.table.toDrive(
        collection=tile_points(n),
        description=n,
        folder=DRIVE_FOLDER,
        fileFormat='CSV',
        selectors=['longitude', 'latitude', 'crop_frac', 'aridity_index']
    ).start()
print('Started', len(todo), 'tasks')

In [ ]:
import glob, os
import pandas as pd
import geopandas as gpd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
OUT  = f'{BASE}/crop_project'
os.makedirs(OUT, exist_ok=True)

# 1. Read all tile CSVs (tiles with no data produce no file or an empty one)
files = sorted(glob.glob(f'{BASE}/{DRIVE_FOLDER}/*.csv'))
frames = []
for f in files:
    d = pd.read_csv(f)
    if len(d):
        frames.append(d[['longitude', 'latitude', 'crop_frac', 'aridity_index']])
print(len(files), 'CSV files,', len(frames), 'non-empty')

df = pd.concat(frames, ignore_index=True)
df = df.rename(columns={'longitude': 'lon', 'latitude': 'lat'})
df['grid_id'] = df['lon'].round(4).astype(str) + '_' + df['lat'].round(4).astype(str)
n_before = len(df)
df = df.drop_duplicates('grid_id').reset_index(drop=True)
print(f'Rows: {n_before} -> {len(df)} after removing duplicates')

# 2. State boundaries from Earth Engine (simplified), used for labelling and the app
admin1 = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.inList('ADM0_NAME', COUNTRIES)))
simple = (admin1.map(lambda f: f.simplify(1000))
          .select(['ADM0_NAME', 'ADM1_NAME', 'ADM1_CODE']))
gdf = gpd.GeoDataFrame.from_features(simple.getInfo()['features'], crs='EPSG:4326')
gdf = gdf.rename(columns={'ADM0_NAME': 'adm0', 'ADM1_NAME': 'adm1', 'ADM1_CODE': 'adm1_code'})

# 3. Label each grid point with country and state
pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs='EPSG:4326')
joined = gpd.sjoin(pts, gdf[['adm0', 'adm1', 'adm1_code', 'geometry']],
                   how='inner', predicate='within')
joined = (joined.drop(columns=['index_right', 'geometry'])
                .drop_duplicates('grid_id').reset_index(drop=True))
print(f'Dropped {len(df) - len(joined)} points outside the nine countries')

out = joined.astype({'lon': 'float32', 'lat': 'float32',
                     'crop_frac': 'float32', 'aridity_index': 'float32',
                     'adm1_code': 'int32'})

# 4. Save
out.to_parquet(f'{OUT}/grid.parquet', index=False)
gdf.to_file(f'{OUT}/states.geojson', driver='GeoJSON')
gdf.to_file(f'{OUT}/states.gpkg', driver='GPKG')

print('\nFinal grid:', out.shape)
print(out.groupby('adm0').size().sort_values(ascending=False))
print(out.describe().loc[['min', 'max']])